# Lab02：episode windows、mask 与因果对齐

从合成 episodes 构造 history/action chunks，检查未来 force 泄漏与 train-only normalization。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import json
from pathlib import Path
import torch


### 定义：`Episode`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
@dataclass
class Episode:
    episode_id: int
    time: torch.Tensor
    proprio: torch.Tensor
    action: torch.Tensor
    force_available_time: torch.Tensor
    force: torch.Tensor


### 定义：`make_episode`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def make_episode(episode_id: int, length: int) -> Episode:
    time = torch.arange(length, dtype=torch.float64) / 10.0
    proprio = torch.stack((time + 0.2 * episode_id, torch.sin(time)), dim=-1).float()
    action = proprio + torch.tensor([0.1, -0.05])
    # Force is available every 20 ms, with an offset relative to the 10 Hz steps.
    force_time = torch.arange(length * 5, dtype=torch.float64) / 50.0 - 0.015
    force = (2 * torch.sin(3 * force_time) + episode_id).float().unsqueeze(-1)
    return Episode(episode_id, time, proprio, action, force_time, force)


### 定义：`aligned_observation`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def aligned_observation(episode: Episode) -> tuple[torch.Tensor, torch.Tensor]:
    indices = torch.searchsorted(episode.force_available_time, episode.time, right=True) - 1
    if (indices < 0).any():
        raise ValueError("Some control steps have no available force sample.")
    selected_time = episode.force_available_time[indices]
    if (selected_time > episode.time).any():
        raise ValueError("Future force leakage.")
    return torch.cat((episode.proprio, episode.force[indices]), dim=-1), selected_time


### 定义：`make_window`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def make_window(episode: Episode, step: int, observation_horizon: int, action_horizon: int) -> dict:
    observation, force_time = aligned_observation(episode)
    history_indices = torch.arange(step - observation_horizon + 1, step + 1)
    observation_mask = history_indices >= 0
    safe_history = history_indices.clamp(min=0)
    action_indices = torch.arange(step, step + action_horizon)
    action_mask = action_indices < len(episode.time)
    safe_action = action_indices.clamp(max=len(episode.time) - 1)
    chunk = episode.action[safe_action].clone()
    chunk[~action_mask] = 0
    return {
        "episode_id": episode.episode_id,
        "step": step,
        "observation": observation[safe_history],
        "observation_mask": observation_mask,
        "observation_time": episode.time[safe_history],
        "force_available_time": force_time[safe_history],
        "action": chunk,
        "action_mask": action_mask,
    }


## 2. 生成并按 episode split


In [ ]:
torch.set_num_threads(1)
episodes = [make_episode(i, length) for i, length in enumerate((9, 13, 7, 11, 8, 12))]
train, validation = episodes[:4], episodes[4:]
train_ids = {e.episode_id for e in train}
validation_ids = {e.episode_id for e in validation}
if train_ids & validation_ids:
    raise ValueError("Episode split leakage.")

# Each genuine training action contributes once; validation and padding do not.


## 3. 只用真实 train actions 拟合统计


In [ ]:
train_actions = torch.cat([episode.action for episode in train])
action_mean = train_actions.mean(dim=0)
action_std = train_actions.std(dim=0, correction=0).clamp(min=1e-6)
all_actions = torch.cat([episode.action for episode in episodes])


## 4. 构造 windows 与 padding masks


In [ ]:
observation_horizon, action_horizon = 3, 4
windows = [
    make_window(episode, step, observation_horizon, action_horizon)
    for episode in train
    for step in range(len(episode.time))
]
observation = torch.stack([window["observation"] for window in windows])
observation_mask = torch.stack([window["observation_mask"] for window in windows])
action = torch.stack([window["action"] for window in windows])
action_mask = torch.stack([window["action_mask"] for window in windows])
observation_time = torch.stack([window["observation_time"] for window in windows])
force_time = torch.stack([window["force_available_time"] for window in windows])
force_age = (observation_time - force_time)[observation_mask]


## 5. Normalization round trip 与 masked loss


In [ ]:
normalized = (action - action_mean) / action_std
normalized[~action_mask] = 0
recovered = normalized * action_std + action_mean
round_trip_error = (recovered[action_mask] - action[action_mask]).abs().max().item()
if round_trip_error > 1e-6:
    raise ValueError("Action normalization round trip failed.")

# A zero prediction illustrates the masked reduction, without training a model.
element_loss = normalized.square()
valid_elements = int(action_mask.sum()) * action.shape[-1]
masked_loss = (element_loss * action_mask.unsqueeze(-1)).sum() / valid_elements


## 6. 汇总检查与边界样本


In [ ]:
result = {
    "config": {"observation_horizon": observation_horizon, "action_horizon": action_horizon},
    "data_contract": {
        "observation": "two synthetic proprio values plus one scalar force; causal availability times",
        "action": "two synthetic continuous target coordinates, not a real robot action interface",
        "padding": "repeat-first history with mask; zero action padding with mask",
        "split": "episode IDs; train and validation only in this teaching example",
        "limitation": "No physical dynamics, learned policy, frame transforms or robot success evaluation.",
    },
    "train_episode_ids": sorted(train_ids),
    "validation_episode_ids": sorted(validation_ids),
    "n_train_actions_for_statistics": len(train_actions),
    "shapes": {"observation": list(observation.shape), "action": list(action.shape)},
    "action_normalization": {
        "train_mean": action_mean.tolist(),
        "train_std": action_std.tolist(),
        "all_data_mean_for_comparison_only": all_actions.mean(dim=0).tolist(),
        "max_valid_round_trip_error": round_trip_error,
    },
    "invalid_history_slots": int((~observation_mask).sum()),
    "invalid_action_slots": int((~action_mask).sum()),
    "force_age_seconds": {"min": force_age.min().item(), "max": force_age.max().item()},
    "future_force_sample_count": int(((force_time > observation_time) & observation_mask).sum()),
    "masked_normalized_mse_of_zero_prediction": masked_loss.item(),
    "boundary_examples": [],
}
for window in (windows[0], windows[-1]):
    result["boundary_examples"].append({
        "episode_id": window["episode_id"],
        "step": window["step"],
        "observation_mask": window["observation_mask"].tolist(),
        "action_mask": window["action_mask"].tolist(),
        "action_chunk": window["action"].tolist(),
    })


In [ ]:
results = result
assert result['future_force_sample_count'] == 0
print(json.dumps(results, indent=2))
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(observation_mask[:12].numpy(), aspect='auto', vmin=0, vmax=1)
axes[0].set(title='Observation valid mask', xlabel='history slot', ylabel='window')
axes[1].imshow(action_mask[-12:].numpy(), aspect='auto', vmin=0, vmax=1)
axes[1].set(title='Action valid mask', xlabel='future slot', ylabel='window')
plt.tight_layout(); plt.show()


## 面试追问与实验记录
1. 为什么统计不能从重叠 windows 重复计数、也不能读取 validation？
2. 把对齐规则改成最近时间点，何时会用到未来 force？
3. padding loss 的分母应该是什么？本实验没有 image pipeline 或真实 frame conversion。

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
